# Independent grading of an actual native E2B episode

Use **Accelerator None**, Internet off and the original organizer environment.
Attach the official competition data, organizer wheelhouse, private NeuralSWE
Offline Test Dependencies input and one actual episode input with
`grade_request.json` plus its hash-bound files. Do not attach reference fixes to
the Android agent. This notebook loads no model and cannot improve or establish
the required-31B leaderboard score.

Run only after a real device/emulator episode is recovered and the unchanged
patch is reconstructed. No sample patch or fabricated episode is supplied.
The public-development task, all failures and the full grading conditions stay
recorded. Clean-base and reference controls never count as generated repairs.
The supplemental dependency profile is the genuine legacy profile previously
tested; its equivalence to the private scoring environment remains unproven.

An independent grading pass does not prove native tool autonomy, app interaction
quality, predictor superiority or compliance with a full workflow deadline.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, base64, zipfile, io, subprocess, importlib, glob
from datetime import datetime, timezone
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_MOBILE_VALIDATION = True
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')


In [ ]:
if IS_KAGGLE and RUN_MOBILE_VALIDATION:
    if not WHEELHOUSE_DIR.is_dir():
        raise RuntimeError('Attach the organizer wheelhouse before execution')
    for key, value in {'LITELLM_LOCAL_MODEL_COST_MAP':'True', 'TRANSFORMERS_NO_TF':'1',
        'VLLM_WORKER_MULTIPROC_METHOD':'spawn', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS':'1',
        'VLLM_ENGINE_READY_TIMEOUT_S':'1200', 'VLLM_NO_USAGE_STATS':'1',
        'OTEL_SDK_DISABLED':'true', 'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}.items():
        os.environ[key] = value
    # Organizer workaround for broken cutlass hooks, restricted to that package's .pth files.
    for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                    '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
        for path in glob.glob(pattern): Path(path).unlink(missing_ok=True)
    wheel_dir = Path('/tmp/neural_swe_wheelhouse')
    wheel_dir.mkdir(exist_ok=True)
    for wheel in WHEELHOUSE_DIR.glob('*.whl'):
        if 'cutlass' in wheel.name.lower(): continue
        name = wheel.name.replace('cu128', '+cu128') if 'cu128' in wheel.name and '+' not in wheel.name else wheel.name
        link = wheel_dir / name
        if not link.exists(): link.symlink_to(wheel)
    wheels = sorted(str(p) for p in wheel_dir.glob('*.whl'))
    if not wheels: raise RuntimeError('Empty organizer wheelhouse')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                    '--no-deps', '--force-reinstall', *wheels], check=True)
    importlib.invalidate_caches()


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
from pathlib import Path
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifact_paths = sorted(p for p in run_root.rglob('*') if p.is_file()
                            and not p.is_symlink() and p.name != 'evidence_bundle.zip')
    manifest = []
    for artifact in artifact_paths:
        resolved = artifact.resolve()
        if not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact escaped its directory')
        manifest.append({'path': artifact.relative_to(run_root).as_posix(),
                         'bytes': artifact.stat().st_size,
                         'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for artifact in artifact_paths:
            archive.write(artifact, artifact.relative_to(run_root).as_posix())
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    (run_root / 'evidence_bundle.zip').write_bytes(bundle_bytes)
    report = {'sha256': hashlib.sha256(bundle_bytes).hexdigest(), 'bytes': len(bundle_bytes), 'file_count': len(manifest)}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
"""Independent official CPU grading of unchanged native repository exports.

Definitions do not execute inference or grading. Public reference solutions stay
evaluator-only; neither references nor clean-base passes are agent repair wins.
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import json
from pathlib import Path, PurePosixPath
import time

EXPECTED_CORE_PACKAGES = {"swegemma": "0.2.7", "adk-submission": "0.2.12", "adk-eval-core": "0.1.0"}
TASKS_SHA256 = "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6"
GUIDE_SHA256 = "3d6e57a13234cb4e783ba24eaab486459af76e0923ea4c6607cd41cc8961bbbb"
VERIFY_TASK_SHA256 = "98600fd6066a4dbd472490cb194b839a3a633afea9b99256cc081ea77beac0d3"


def sha_file(path):
    value = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(1048576):
            value.update(chunk)
    return value.hexdigest()


def installed_versions(names):
    values = {}
    for name in names:
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def bound_file(root, name):
    relative = PurePosixPath(name)
    if (not name or "\\" in name or ":" in name or relative.is_absolute()
            or any(part in {"", ".", ".."} for part in name.split("/"))):
        raise ValueError("Invalid evidence member path")
    result = root.joinpath(*relative.parts)
    if result.is_symlink() or not result.resolve().is_relative_to(root.resolve()):
        raise ValueError("Evidence member escaped its root")
    return result


def validate_request(root, request):
    if request["format"] != "native_mobile_grade_request_v1" or request["scope"] != "public_development_not_held_out":
        raise ValueError("Unexpected grading request scope")
    if (request["expected_core_packages"] != EXPECTED_CORE_PACKAGES
            or request["test_timeout_seconds"] != 300 or request["tasks_sha256"] != TASKS_SHA256
            or request["guide_sha256"] != GUIDE_SHA256 or request["expected_verify_task_sha256"] != VERIFY_TASK_SHA256):
        raise ValueError("Grading request differs from the observed organizer contract")
    for name, expected in request["source_sha256"].items():
        if sha_file(bound_file(root, name)) != expected:
            raise ValueError("Mobile evidence changed: " + name)
    def read(name):
        if name not in request["source_sha256"]:
            raise ValueError("Unbound mobile evidence member")
        return json.loads(bound_file(root, name).read_text())
    assignment = read("assignment.json")
    summary = read("summary.json")
    episode = read("episode.json")
    reconstruction = read(request["reconstruction_path"])
    task_id = request["task_id"]
    if (assignment["task_id"] != task_id or summary["task_id"] != task_id or episode["task_id"] != task_id
            or assignment["run_id"] != request["run_id"] or episode["run_id"] != request["run_id"]
            or summary["run_id"] != request["run_id"]
            or summary["run_identity_matches"] is not True
            or reconstruction["task_id"] != task_id
            or reconstruction["repo"] != request["repo"] or reconstruction["base_commit"] != request["base_commit"]
            or reconstruction["input_manifest_sha256"] != assignment["input_manifest_sha256"]
            or assignment["input_manifest"] != episode["input_manifest"]
            or (assignment["input_manifest"]["task_id"], assignment["input_manifest"]["repo"], assignment["input_manifest"]["base_commit"]) !=
               (task_id, request["repo"], request["base_commit"])
            or reconstruction["reconstructed_tree_matches_device_inventory"] is not True):
        raise ValueError("Native assignment/export identities differ")
    patch_path = request["patch_path"]
    if patch_path not in request["source_sha256"]:
        raise ValueError("Unbound generated patch")
    patch = bound_file(root, patch_path).read_bytes()
    if hashlib.sha256(patch).hexdigest() != reconstruction["patch_sha256"] or len(patch) != reconstruction["patch_bytes"]:
        raise ValueError("Generated patch differs from reconstructed device bytes")
    if reconstruction.get("export_origin") == "host_recovery_of_quiesced_native_workspace_not_native_export_completion":
        recovery = read("recovery/recovery_manifest.json")
        required = {"workspace_after_stop.tar", "recovery/recovered_changes.zip"}
        if not required.issubset(request["source_sha256"]):
            raise ValueError("Recovered working bytes are not bound to grading")
        if (assignment.get("workspace_recovery_quiesced") is not True
                or recovery["process_quiesced_by_runner"] is not True
                or recovery["run_id"] != request["run_id"] or recovery["task_id"] != task_id
                or recovery["native_episode_sha256"] != request["source_sha256"]["episode.json"]
                or recovery["input_manifest_sha256"] != assignment["input_manifest_sha256"]
                or recovery["workspace_tar_sha256"] != request["source_sha256"]["workspace_after_stop.tar"]
                or recovery["recovered_export_sha256"] != request["source_sha256"]["recovery/recovered_changes.zip"]
                or reconstruction["device_export_sha256"] != recovery["recovered_export_sha256"]
                or recovery["generation_completion_established_by_recovery"] is not False):
            raise ValueError("Recovered workspace provenance differs from its native assignment")
    return patch.decode("utf-8"), assignment, summary


def classify_patch(clean, candidate, reference, patch_bytes):
    def passed(row):
        return row.get("resolved") is True and row.get("test_exit_code") == 0
    known_fixture = passed(reference) and clean.get("resolved") is False and clean.get("test_exit_code") in (1, 2)
    return {"generated_patch_passes_official_tests": passed(candidate),
            "clean_base_passes": passed(clean), "reference_passes": passed(reference),
            "fixture_discriminates": known_fixture,
            "discriminating_patch_success": bool(known_fixture and passed(candidate) and patch_bytes > 0),
            "model_inference_established_by_grading": False,
            "full_workflow_deadline_verified": None, "official_leaderboard_score": None}


def run_mobile_patch_validation(data_dir, episode_root, run_root, request, checkpoint_exporter, environment_receipt=None):
    patch, assignment, native_summary = validate_request(episode_root, request)
    if sha_file(data_dir / "tasks.jsonl") != request["tasks_sha256"] or sha_file(data_dir / "HARNESS_README.md") != request["guide_sha256"]:
        raise ValueError("Organizer data differs from frozen grading request")
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    if torch.cuda.device_count() != 0:
        raise ValueError("Select Accelerator None: this is CPU patch validation, not model inference")
    packages = {name: importlib.metadata.version(name) for name in EXPECTED_CORE_PACKAGES}
    if packages != request["expected_core_packages"]:
        raise ValueError("Organizer core package versions differ from frozen request")
    task = next(item for item in load_tasks(data_dir / "tasks.jsonl") if item.instance_id == request["task_id"])
    if task.repo != request["repo"] or task.base_commit != request["base_commit"]:
        raise ValueError("Organizer task differs from native assignment")
    # Read the reference only in the evaluator after generation is frozen. Never
    # print or export its text, and never count it as a generated solution.
    reference_row = next(row for row in map(json.loads, (data_dir / "tasks.jsonl").read_text().splitlines())
                         if row["instance_id"] == request["task_id"])
    reference_patch = reference_row["patch"]
    module = inspect.getmodule(Evaluator)
    run_root.mkdir(parents=True, exist_ok=False)
    verifier_source = inspect.getsource(module.verify_task)
    (run_root / "verify_task_api.py").write_text(verifier_source, encoding="utf-8")
    if hashlib.sha256(verifier_source.encode()).hexdigest() != request["expected_verify_task_sha256"]:
        raise ValueError("Official verifier source differs from the recorded real CPU fixture run")
    snapshot, base_snapshot, task_patch = module.resolve_task_snapshot_paths(data_dir / "snapshots", task.instance_id, task.repo)
    snapshots = {str(path.relative_to(data_dir)): sha_file(path) for path in (snapshot, base_snapshot, task_patch) if path is not None}
    report = {"scope": "official_public_cpu_validation_of_frozen_mobile_patch_not_leaderboard",
              "created_utc": datetime.now(timezone.utc).isoformat(), "run_id": request["run_id"],
              "task_id": task.instance_id, "assigned_native_tasks": 1, "assigned_grading_conditions": 3,
              "grading_request": request, "status": "started", "observations": [],
              "generator_reported_status": native_summary,
              "model_inference_executed_by_grader": False, "gpu_count": 0,
              "core_packages": packages, "official_snapshot_sha256": snapshots,
              "generator_input_origin": "GitHub_codeload_exact_declared_commit",
              "device_input_tree_equivalence_with_official_snapshot_established": False,
              "private_scoring_environment_equivalence_established": False,
              "supplemental_environment_receipt": environment_receipt,
              "observed_test_packages": installed_versions(("pytest", "Flask", "Werkzeug", "requests", "httpbin", "pytest-httpbin")),
              "verify_task_source_sha256": hashlib.sha256(verifier_source.encode()).hexdigest(),
              "task_correctness": None, "full_workflow_deadline_verified": None,
              "reference_text_exported": False, "official_leaderboard_score": None}
    def save():
        (run_root / "validation.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    save(); checkpoint_exporter(run_root)

    async def evaluate():
        outcomes = {}
        for condition, submitted_patch in (("clean_base", ""), ("native_unchanged_patch", patch), ("official_reference", reference_patch)):
            destination = run_root / condition
            destination.mkdir()
            config = EvalConfig(tasks_path=data_dir / "tasks.jsonl", snapshots_dir=data_dir / "snapshots",
                results_dir=destination, submission_dir=run_root / "unused_agent_directory", models={},
                sandbox="subprocess", timeout_seconds=request["test_timeout_seconds"], wheels_dir=data_dir / "wheels",
                graph_dir=str(data_dir / "graphs"), embeddings_dir=str(data_dir / "embeddings"))
            evaluator = Evaluator(config)
            print("MOBILE_PATCH_VALIDATION_START " + condition, flush=True)
            result = await module.verify_task(evaluator.docker, config, task, snapshot,
                base_snapshot_path=base_snapshot, patch_path=task_patch,
                agent_patch=submitted_patch, start_time=time.perf_counter())
            details = result.model_dump(mode="json", exclude={"trace", "agent_patch"})
            details.update(condition=condition, replayed_patch_sha256=hashlib.sha256(submitted_patch.encode()).hexdigest())
            (destination / "official_task_result.json").write_text(json.dumps(details, indent=2) + "\n", encoding="utf-8")
            (destination / "test_output.txt").write_text(result.test_output or "", encoding="utf-8")
            observed = {"condition": condition, "resolved": result.resolved, "test_exit_code": result.test_exit_code,
                        "duration_seconds": result.duration_seconds, "result_path": condition + "/official_task_result.json"}
            outcomes[condition] = observed
            report["observations"].append(observed)
            save(); print("MOBILE_PATCH_VALIDATION_RESULT " + json.dumps(observed), flush=True)
            checkpoint_exporter(run_root)
        report["classification"] = classify_patch(outcomes["clean_base"], outcomes["native_unchanged_patch"],
                                                    outcomes["official_reference"], len(patch.encode()))
        report["status"] = "completed"
        report["task_correctness"] = report["classification"]["generated_patch_passes_official_tests"]
        save(); checkpoint_exporter(run_root)
        return report
    try:
        with ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(evaluate())).result()
    except Exception as error:
        report.update(status="grading_exception", error_type=type(error).__name__, error=str(error))
        save(); checkpoint_exporter(run_root)
        raise


def prepare_grade_request(episode_root):
    """Freeze a request only from a recovered, hash-bound real episode receipt."""
    assignment = json.loads((episode_root / "assignment.json").read_text())
    summary = json.loads((episode_root / "summary.json").read_text())
    if summary["run_identity_matches"] is not True or summary["instrumentation_started"] is not True:
        raise ValueError("No recovered matching native episode exists")
    reconstruction_dirs = [name for name in ("export_reconstruction", "partial_export_reconstruction", "recovery/reconstruction")
                           if (episode_root / name / "reconstruction.json").is_file()]
    if len(reconstruction_dirs) != 1:
        raise ValueError("Require one unchanged reconstructed device export")
    directory = reconstruction_dirs[0]
    reconstruction = json.loads((episode_root / directory / "reconstruction.json").read_text())
    names = ["assignment.json", "summary.json", "episode.json", directory + "/reconstruction.json", directory + "/generated.patch"]
    names += [name for name in ("memory_samples.jsonl", "model_responses.jsonl", "tool_results.jsonl", "stage_delta.jsonl", "events.jsonl")
              if (episode_root / name).is_file()]
    if directory == "recovery/reconstruction":
        names += ["recovery/recovery_manifest.json", "recovery/recovered_changes.zip", "workspace_after_stop.tar"]
    request = {"format": "native_mobile_grade_request_v1", "scope": "public_development_not_held_out",
               "run_id": assignment["run_id"], "task_id": assignment["task_id"],
               "repo": reconstruction["repo"], "base_commit": reconstruction["base_commit"],
               "source_sha256": {name: sha_file(episode_root / name) for name in names},
               "patch_path": directory + "/generated.patch", "reconstruction_path": directory + "/reconstruction.json",
               "tasks_sha256": TASKS_SHA256, "guide_sha256": GUIDE_SHA256,
               "expected_core_packages": EXPECTED_CORE_PACKAGES, "test_timeout_seconds": 300,
               "expected_verify_task_sha256": VERIFY_TASK_SHA256}
    validate_request(episode_root, request)
    target = episode_root / "grade_request.json"
    with target.open("x", encoding="utf-8") as stream:
        stream.write(json.dumps(request, indent=2) + "\n")
    return request


In [ ]:
if IS_KAGGLE and RUN_MOBILE_VALIDATION:
    import shutil
    locations = list(Path('/kaggle/input').rglob('grade_request.json'))
    if len(locations) != 1:
        raise RuntimeError('Attach exactly one actual recovered native episode input')
    EPISODE_INPUT = locations[0].parent
    REQUEST = json.loads(locations[0].read_text())
    validate_request(EPISODE_INPUT, REQUEST)
    RUN = Path('/kaggle/working') / ('native_patch_validation_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    EPISODE_ROOT = RUN / 'frozen_mobile_evidence'
    EPISODE_ROOT.mkdir()
    shutil.copyfile(locations[0], EPISODE_ROOT / 'grade_request.json')
    for name in REQUEST['source_sha256']:
        target = bound_file(EPISODE_ROOT, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(bound_file(EPISODE_INPUT, name), target)
    validate_request(EPISODE_ROOT, REQUEST)
    export_run(RUN)  # Freeze assigned input evidence before test setup.
else:
    print('Definitions only; no mobile inference or independent grading executed')


In [ ]:
"""Install genuine test dependencies offline without replacing the verifier."""
import hashlib
import importlib
import importlib.metadata as metadata
import io
import json
from pathlib import Path
import platform
import subprocess
import sys
import zipfile


def verify_dependency_bundle(payload: bytes, manifest: dict) -> dict[str, bytes]:
    if hashlib.sha256(payload).hexdigest() != manifest['bundle_sha256']:
        raise ValueError('Dependency bundle hash mismatch')
    expected = {row['file']: row for row in manifest['files']}
    if len(expected) != len(manifest['files']):
        raise ValueError('Duplicate dependency manifest member')
    if any(Path(name).name != name or '/' in name or '\\' in name or not name.endswith('.whl')
           for name in expected):
        raise ValueError('Dependency wheels must be plain basenames')
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(expected):
            raise ValueError('Dependency archive/manifest member mismatch')
        if sum(item.file_size for item in archive.infolist()) > 16 * 1024 * 1024:
            raise ValueError('Unexpected dependency archive size')
        result = {}
        for item in archive.infolist():
            if item.is_dir() or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Invalid dependency archive member')
            data = archive.read(item.filename)
            row = expected[item.filename]
            if len(data) != row['bytes'] or hashlib.sha256(data).hexdigest() != row['sha256']:
                raise ValueError('Dependency wheel identity mismatch: ' + item.filename)
            result[item.filename] = data
        return result


def install_offline_test_dependencies(payload: bytes, manifest: dict, run_root: Path, checkpoint_exporter) -> dict:
    if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'AMD64'):
        raise ValueError('This dependency bundle targets the Kaggle Linux x86_64 runtime')
    if sys.version_info[:2] not in ((3, 12), (3, 13)):
        raise ValueError('This bundle contains wheels for Python 3.12 and 3.13 only')
    wheels = verify_dependency_bundle(payload, manifest)
    profiles = {
        'httpbin_current': ['pytest-httpbin==2.1.0', 'httpbin==0.10.4'],
        'httpbin_legacy_redirect': ['pytest-httpbin==2.1.0', 'httpbin==0.10.4',
                                   'Flask==2.2.5', 'Werkzeug==2.2.3'],
    }
    profile = manifest.get('profile', 'httpbin_current')
    requirements = profiles.get(profile)
    if requirements is None or manifest['requirements'] != requirements:
        raise ValueError('Unexpected test-environment requirements')
    stage = Path('/tmp') / (run_root.name + '_dependency_wheels')
    stage.mkdir(exist_ok=False)
    for name, data in wheels.items():
        (stage / name).write_bytes(data)
    packages = sorted({row['package'] for row in manifest['files']} |
                      {'swegemma', 'adk-submission', 'adk-eval-core', 'vllm', 'torch', 'pytest'})
    def versions():
        result = {}
        for name in packages:
            try:
                result[name] = metadata.version(name)
            except metadata.PackageNotFoundError:
                result[name] = None
        return result
    before = versions()
    report = {'scope':'supplemental_offline_test_environment_not_agent_or_private_scoring',
              'python_version':sys.version, 'platform':platform.platform(),
              'bundle_sha256':manifest['bundle_sha256'], 'requirements':requirements, 'profile':profile,
              'before':before, 'model_inference_executed':False,
              'organizer_verifier_replaced':False, 'test_assertions_changed':False}
    (run_root / 'dependency_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
    process = subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
        '--no-index','--find-links',str(stage),*requirements],capture_output=True,text=True)
    (run_root / 'dependency_install_stdout.txt').write_text(process.stdout,encoding='utf-8')
    (run_root / 'dependency_install_stderr.txt').write_text(process.stderr,encoding='utf-8')
    importlib.invalidate_caches()
    after = versions()
    report.update(return_code=process.returncode,after=after,
        changed_packages={name:{'before':before[name],'after':after[name]}
                          for name in packages if before[name]!=after[name]},
        pytest_entry_points=[{'name':entry.name,'value':entry.value,'distribution':entry.dist.name}
                             for entry in metadata.entry_points(group='pytest11')])
    core_names = ('swegemma','adk-submission','adk-eval-core','vllm','torch','pytest')
    report['organizer_core_versions_unchanged'] = all(before[name]==after[name] for name in core_names)
    report['httpbin_plugin_registered'] = any(entry['value']=='pytest_httpbin.plugin'
                                            for entry in report['pytest_entry_points'])
    report['status'] = ('installed' if process.returncode==0 and report['organizer_core_versions_unchanged']
                        and report['httpbin_plugin_registered'] else 'installation_not_verified')
    (run_root/'dependency_install_report.json').write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
    checkpoint_exporter(run_root)
    if report['status'] != 'installed':
        raise RuntimeError('Offline dependency installation not verified; preserve its real logs')
    print('OFFLINE_TEST_DEPENDENCIES ' + json.dumps(report),flush=True)
    return report


In [ ]:
DEPENDENCY_MANIFEST = json.loads('{"scope": "supplemental_test_dependencies_not_agent_submission", "created_utc": "2026-10-03T19:45:18.893670+00:00", "profile": "httpbin_legacy_redirect", "requirements": ["pytest-httpbin==2.1.0", "httpbin==0.10.4", "Flask==2.2.5", "Werkzeug==2.2.3"], "target": "Linux x86_64 CPython 3.12/3.13", "bundle_sha256": "fe32aa50e4fb0f1c4886e2c320382744d6022d07296035e1b72259f7698769f2", "files": [{"file": "blinker-1.9.0-py3-none-any.whl", "package": "blinker", "version": "1.9.0", "bytes": 8458, "sha256": "ba0efaa9080b619ff2f3459d1d500c57bddea4a6b424b60a91141db6fd2f08bc", "source_url": "https://files.pythonhosted.org/packages/10/cb/f2ad4230dc2eb1a74edf38f1a38b9b52277f75bef262d8908e60d957e13c/blinker-1.9.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/blinker/1.9.0/json", "requires_python": ">=3.9", "requires_dist": []}, {"file": "brotlicffi-1.2.0.2-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "package": "brotlicffi", "version": "1.2.0.2", "bytes": 1536851, "sha256": "22916101de0e7ff535f2edf54b52a85591853b8ae9a98737643defdd3c063a3a", "source_url": "https://files.pythonhosted.org/packages/06/78/076419ed6c2c6aa3eaac6fd6b076502b4be89d50625fcdc513cd4aeca718/brotlicffi-1.2.0.2-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/brotlicffi/1.2.0.2/json", "requires_python": ">=3.8", "requires_dist": ["cffi>=1.0.0; python_version < \\"3.13\\"", "cffi>=1.17.0; python_version >= \\"3.13\\""]}, {"file": "cffi-2.1.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "package": "cffi", "version": "2.1.1", "bytes": 221822, "sha256": "c1453022f490d2459a11819d83ad1d586e9ff65a12ac3e705ffebd46d3685dcf", "source_url": "https://files.pythonhosted.org/packages/b1/db/dceb9dd5b231e1da801793f8acc9f3c52a7e1afe40bb1aae37e02b0faad5/cffi-2.1.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/cffi/2.1.1/json", "requires_python": ">=3.10", "requires_dist": ["pycparser; implementation_name != \\"PyPy\\""]}, {"file": "cffi-2.1.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "package": "cffi", "version": "2.1.1", "bytes": 221824, "sha256": "a931079504ecc49efed7744c476a5c343a92fabf66dec2db95edb1b2fdc770e2", "source_url": "https://files.pythonhosted.org/packages/95/95/86342356ff5953b3fb06f7ef7c5bee212d45e770abc7218d451b9148313c/cffi-2.1.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/cffi/2.1.1/json", "requires_python": ">=3.10", "requires_dist": ["pycparser; implementation_name != \\"PyPy\\""]}, {"file": "click-8.5.0-py3-none-any.whl", "package": "click", "version": "8.5.0", "bytes": 125251, "sha256": "255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360", "source_url": "https://files.pythonhosted.org/packages/58/50/6c0d534c5f134586a8e1ba4e330569e32f057e33372ae556463212fb4cd3/click-8.5.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/click/8.5.0/json", "requires_python": ">=3.10", "requires_dist": []}, {"file": "decorator-5.3.1-py3-none-any.whl", "package": "decorator", "version": "5.3.1", "bytes": 10365, "sha256": "f47fe6fdbd2edd623ecfe36875d37aba411624e2670dd395dddae1358689bb3c", "source_url": "https://files.pythonhosted.org/packages/05/7f/798705f5296a58ca505d600456748d1be48078eac8a7050d8a98bc9edb89/decorator-5.3.1-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/decorator/5.3.1/json", "requires_python": ">=3.8", "requires_dist": []}, {"file": "Flask-2.2.5-py3-none-any.whl", "package": "Flask", "version": "2.2.5", "bytes": 101817, "sha256": "58107ed83443e86067e41eff4631b058178191a355886f8e479e347fa1285fdf", "source_url": "https://files.pythonhosted.org/packages/9f/1a/8b6d48162861009d1e017a9740431c78d860809773b66cac220a11aa3310/Flask-2.2.5-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/Flask/2.2.5/json", "requires_python": ">=3.7", "requires_dist": ["Werkzeug (>=2.2.2)", "Jinja2 (>=3.0)", "itsdangerous (>=2.0)", "click (>=8.0)", "importlib-metadata (>=3.6.0) ; python_version < \\"3.10\\"", "asgiref (>=3.2) ; extra == \'async\'", "python-dotenv ; extra == \'dotenv\'"]}, {"file": "greenlet-3.5.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "package": "greenlet", "version": "3.5.6", "bytes": 621404, "sha256": "975736b002ed080d124cf81a79cb7e05cb26d6b3f5c7a7b651c0fcce70353aa1", "source_url": "https://files.pythonhosted.org/packages/66/6a/1594f3869c57c149abdb380492529e04d4c0229b5e4d79572c5bd0aaa673/greenlet-3.5.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/greenlet/3.5.6/json", "requires_python": ">=3.10", "requires_dist": ["Sphinx; extra == \\"docs\\"", "furo; extra == \\"docs\\"", "objgraph; extra == \\"test\\"", "psutil; extra == \\"test\\"", "setuptools; extra == \\"test\\""]}, {"file": "greenlet-3.5.6-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "package": "greenlet", "version": "3.5.6", "bytes": 623779, "sha256": "f98e8215e172f567ce80eeaed9107fb4d32b6c44f26983d9b8334658136a205a", "source_url": "https://files.pythonhosted.org/packages/85/cb/ab0c123c514ed4e94c0dc9ee2e86362633e6b998cfc05de7fc9ac2eb9690/greenlet-3.5.6-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/greenlet/3.5.6/json", "requires_python": ">=3.10", "requires_dist": ["Sphinx; extra == \\"docs\\"", "furo; extra == \\"docs\\"", "objgraph; extra == \\"test\\"", "psutil; extra == \\"test\\"", "setuptools; extra == \\"test\\""]}, {"file": "httpbin-0.10.4-py2.py3-none-any.whl", "package": "httpbin", "version": "0.10.4", "bytes": 100765, "sha256": "71862093f371c3a39cfcdcd2f073aaf610b6433f9d4c0130248e676453fca65d", "source_url": "https://files.pythonhosted.org/packages/e5/39/e9fc7334c69f82c307ae861e13704437d31b71514d273061e08db5830b92/httpbin-0.10.4-py2.py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/httpbin/0.10.4/json", "requires_python": ">=3.8", "requires_dist": ["brotlicffi", "decorator", "flask>=2.2.4", "greenlet<3.0; python_version < \\"3.12\\"", "greenlet>=3.0.0a1; python_version >= \\"3.12.0rc0\\"", "importlib-metadata; python_version < \\"3.8\\"", "six", "werkzeug>=2.2.2", "pytest; extra == \\"test\\"", "tox; extra == \\"test\\"", "flasgger; extra == \\"mainapp\\"", "gunicorn; extra == \\"mainapp\\"", "gevent; extra == \\"mainapp\\""]}, {"file": "itsdangerous-2.2.0-py3-none-any.whl", "package": "itsdangerous", "version": "2.2.0", "bytes": 16234, "sha256": "c6242fc49e35958c8b15141343aa660db5fc54d4f13a1db01a3f5891b98700ef", "source_url": "https://files.pythonhosted.org/packages/04/96/92447566d16df59b2a776c0fb82dbc4d9e07cd95062562af01e408583fc4/itsdangerous-2.2.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/itsdangerous/2.2.0/json", "requires_python": ">=3.8", "requires_dist": []}, {"file": "jinja2-3.1.6-py3-none-any.whl", "package": "Jinja2", "version": "3.1.6", "bytes": 134899, "sha256": "85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67", "source_url": "https://files.pythonhosted.org/packages/62/a1/3d680cbfd5f4b8f15abc1d571870c5fc3e594bb582bc3b64ea099db13e56/jinja2-3.1.6-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/Jinja2/3.1.6/json", "requires_python": ">=3.7", "requires_dist": ["MarkupSafe>=2.0", "Babel>=2.7 ; extra == \\"i18n\\""]}, {"file": "markupsafe-3.0.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "package": "MarkupSafe", "version": "3.0.4", "bytes": 22985, "sha256": "8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581", "source_url": "https://files.pythonhosted.org/packages/4f/a7/aeedb5140afa41fc74c225e9184ab96723a6e873b6ee1c9fede7283456d8/markupsafe-3.0.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/MarkupSafe/3.0.4/json", "requires_python": ">=3.9", "requires_dist": []}, {"file": "markupsafe-3.0.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "package": "MarkupSafe", "version": "3.0.4", "bytes": 23017, "sha256": "434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6", "source_url": "https://files.pythonhosted.org/packages/63/e0/cec6865dfe88cb48fedd4b20aed6af5158e41092adcbf3e028bcc6ec2108/markupsafe-3.0.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "registry_record_url": "https://pypi.org/pypi/MarkupSafe/3.0.4/json", "requires_python": ">=3.9", "requires_dist": []}, {"file": "pycparser-3.0-py3-none-any.whl", "package": "pycparser", "version": "3.0", "bytes": 48172, "sha256": "b727414169a36b7d524c1c3e31839a521725078d7b2ff038656844266160a992", "source_url": "https://files.pythonhosted.org/packages/0c/c3/44f3fbbfa403ea2a7c779186dc20772604442dde72947e7d01069cbe98e3/pycparser-3.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/pycparser/3.0/json", "requires_python": ">=3.10", "requires_dist": []}, {"file": "pytest_httpbin-2.1.0-py3-none-any.whl", "package": "pytest-httpbin", "version": "2.1.0", "bytes": 8309, "sha256": "b3bf7346cc2ad231447189cd85b458e341056684a7a69d69533dd29692209cdd", "source_url": "https://files.pythonhosted.org/packages/0d/97/149289c7123f75e98edc94872545c2f597f349ee037b1e0a47aef0362fd4/pytest_httpbin-2.1.0-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/pytest-httpbin/2.1.0/json", "requires_python": ">=3.8", "requires_dist": ["httpbin", "requests; extra == \\"test\\"", "pytest; extra == \\"test\\""]}, {"file": "six-1.17.0-py2.py3-none-any.whl", "package": "six", "version": "1.17.0", "bytes": 11050, "sha256": "4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274", "source_url": "https://files.pythonhosted.org/packages/b7/ce/149a00dd41f10bc29e5921b496af8b574d8413afcd5e30dfa0ed46c2cc5e/six-1.17.0-py2.py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/six/1.17.0/json", "requires_python": ">=2.7, !=3.0.*, !=3.1.*, !=3.2.*", "requires_dist": []}, {"file": "Werkzeug-2.2.3-py3-none-any.whl", "package": "Werkzeug", "version": "2.2.3", "bytes": 233551, "sha256": "56433961bc1f12533306c624f3be5e744389ac61d722175d543e1751285da612", "source_url": "https://files.pythonhosted.org/packages/f6/f8/9da63c1617ae2a1dec2fbf6412f3a0cfe9d4ce029eccbda6e1e4258ca45f/Werkzeug-2.2.3-py3-none-any.whl", "registry_record_url": "https://pypi.org/pypi/Werkzeug/2.2.3/json", "requires_python": ">=3.7", "requires_dist": ["MarkupSafe (>=2.1.1)", "watchdog ; extra == \'watchdog\'"]}], "verifier_or_test_assertion_changes": false, "dependency_rationale": "Werkzeug 2.2.3 preserves ASCII Location headers via safe_conversion; Flask 2.2.5 supports that version. The Python 3.12-compatible httpbin plugin remains unchanged. This is a supplemental compatibility profile, not the historical requirements lock or proof of equivalence to private scoring.", "source_references": ["https://raw.githubusercontent.com/psf/requests/0b4d494192de489701d3a2e32acef8fb5d3f042e/requirements-dev.txt", "https://raw.githubusercontent.com/pallets/werkzeug/2.2.3/src/werkzeug/wrappers/response.py", "https://raw.githubusercontent.com/pallets/werkzeug/2.2.3/src/werkzeug/urls.py"]}')
DEPENDENCY_INPUT_PREFIX = 'httpbin_legacy_redirect'
if IS_KAGGLE and RUN_MOBILE_VALIDATION:
    # Attach the private NeuralSWE Offline Test Dependencies dataset. Package
    # binaries are input files, not notebook source (Kaggle limits source to 1 MB).
    matches = list(Path('/kaggle/input').rglob(DEPENDENCY_INPUT_PREFIX+'_manifest.json'))
    if len(matches) != 1:
        raise RuntimeError('Attach exactly one version of the supplemental dependency dataset')
    input_root = matches[0].parent
    if json.loads(matches[0].read_text()) != DEPENDENCY_MANIFEST:
        raise ValueError('Attached dependency manifest differs from the frozen source')
    archive_path = input_root/(DEPENDENCY_INPUT_PREFIX+'_wheels.zip')
    if archive_path.is_file():
        dependency_payload = archive_path.read_bytes()
    else:
        # Kaggle may expand uploaded ZIPs. Recreate the same deterministic
        # transport archive from byte-verified original wheels without alteration.
        stream = io.BytesIO()
        with zipfile.ZipFile(stream,'w',compression=zipfile.ZIP_STORED) as archive:
            for row in DEPENDENCY_MANIFEST['files']:
                locations = list(input_root.rglob(row['file']))
                locations = [p for p in locations if
                    DEPENDENCY_INPUT_PREFIX+'_wheels' in p.parts]
                if len(locations) != 1:
                    raise ValueError('Missing or ambiguous genuine dependency wheel: '+row['file'])
                data = locations[0].read_bytes()
                if len(data)!=row['bytes'] or hashlib.sha256(data).hexdigest()!=row['sha256']:
                    raise ValueError('Attached dependency wheel changed')
                item=zipfile.ZipInfo(row['file'],date_time=(1980,1,1,0,0,0))
                item.create_system=0  # Match the recorded Windows transport metadata.
                item.external_attr=0o100644 << 16
                archive.writestr(item,data)
        dependency_payload = stream.getvalue()
    dependency_report = install_offline_test_dependencies(
        dependency_payload,DEPENDENCY_MANIFEST,RUN,export_run)
else:
    print('Local definitions only; no test-environment installation performed')


In [ ]:
if IS_KAGGLE and RUN_MOBILE_VALIDATION:
    validation = run_mobile_patch_validation(DATA_DIR, EPISODE_ROOT, RUN / 'grading', REQUEST,
        lambda grading_root: export_run(RUN), environment_receipt=dependency_report)
    print(json.dumps(validation, indent=2))
else:
    print('No grading result or competitive gain established')


In [ ]:
if IS_KAGGLE and RUN_MOBILE_VALIDATION:
    export_run(RUN)
